Imports

In [1]:
import sys
import os
package_path = os.path.abspath("../..")  
sys.path.insert(0, package_path)
#The path will be managed by conda or whatever on release, but this is fine for now.
import scMPRAforge as scm
import pandas as pd
import numpy as np

2025-05-28 16:43:47.031075: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-28 16:43:47.036033: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-05-28 16:43:47.036047: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [1]:
#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

In [2]:
#dask imports
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import socket

In [3]:
scm.helloworld()

hello world!


Make the dask cluster & client in accordance with resource avail and model size

In [4]:
cluster=SLURMCluster(
    cores=2,#cores per slurm job
    memory="16G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=2:00:00",
        f"--output=slave_%j.out"]
)
client = Client(cluster,
    timeout=f"{5*60}s",   # Client <-> scheduler timeout 
    heartbeat_interval="20s"  # Worker heartbeat interval
)

In [5]:
cluster.scale(jobs=1)

In [6]:
dat=scm.load_scMPRA_data("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres.tsv")

In [7]:
#let's make an ortho object
test=scm.ortho()
test.criss_cross(client=client,dat=dat,retain_design_matricies=True)
test.extract_params(client)

In [8]:
description=scm.describe_parameters(client,parameters=test.by_cre_parameters.result(),dat=dat,split="cre_id")

In [10]:
scm.undo_one_hot_encoding(description)

,cre_id,cells,nb,zi,theta,r,sigmasquare,p,cell_type,rep_id
0,nobody,503,1.770934,0.455115,0.773889,2.168181,3.217402,0.550423,blood,2
1,nobody,500,1.770934,0.880460,0.773889,2.168181,3.217402,0.550423,blood,3
2,nobody,499,1.770934,0.773005,0.773889,2.168181,3.217402,0.550423,blood,1
3,nobody,443,0.928299,0.455115,0.773889,2.168181,1.325747,0.700208,brain,2
4,nobody,441,0.928299,0.880460,0.773889,2.168181,1.325747,0.700208,brain,3
5,nobody,440,0.928299,0.773005,0.773889,2.168181,1.325747,0.700208,brain,1
6,somebody,522,12.463882,0.519696,1.110941,3.037215,63.612172,0.195935,blood,2
7,somebody,504,12.463882,0.799838,1.110941,3.037215,63.612172,0.195935,blood,1
8,somebody,498,12.463882,0.902793,1.110941,3.037215,63.612172,0.195935,blood,3
9,somebody,464,9.956394,0.799838,1.110941,3.037215,42.594772,0.233747,brain,1


In [28]:
description

,C(cell_type)[blood],C(cell_type)[brain],C(rep_id)[1],C(rep_id)[2],C(rep_id)[3],cre_id,cells,nb,zi,theta,r,sigmasquare,p
0,1,0,0,1,0,nobody,503,1.771708,0.454830,0.775813,2.172357,3.216660,0.550791
1,1,0,0,0,1,nobody,500,1.771708,0.880332,0.775813,2.172357,3.216660,0.550791
2,1,0,1,0,0,nobody,499,1.771708,0.772789,0.775813,2.172357,3.216660,0.550791
3,0,1,0,1,0,nobody,443,0.929128,0.454830,0.775813,2.172357,1.326520,0.700425
4,0,1,0,0,1,nobody,441,0.929128,0.880332,0.775813,2.172357,1.326520,0.700425
5,0,1,1,0,0,nobody,440,0.929128,0.772789,0.775813,2.172357,1.326520,0.700425
6,1,0,0,1,0,somebody,522,12.475343,0.519749,1.110898,3.037085,63.719941,0.195784
7,1,0,1,0,0,somebody,504,12.475343,0.799874,1.110898,3.037085,63.719941,0.195784
8,1,0,0,0,1,somebody,498,12.475343,0.902804,1.110898,3.037085,63.719941,0.195784
9,0,1,1,0,0,somebody,464,9.955950,0.799874,1.110898,3.037085,42.592819,0.233747


In [101]:
import dask.dataframe as dd

In [103]:
working["cre_id"]=working["cre_id"].astype("category")

In [104]:
working=auto_partition(working,50)

In [105]:
working

,C(cell_type)[blood],C(cell_type)[brain],C(rep_id)[1],C(rep_id)[2],C(rep_id)[3],cre_id,cells,nb,zi,theta,r,sigmasquare,p
npartitions=2,,,,,,,,,,,,,
0,int64,int64,int64,int64,int64,category[known],int64,float64,float64,float32,float32,float64,float64
15,...,...,...,...,...,...,...,...,...,...,...,...,...
29,...,...,...,...,...,...,...,...,...,...,...,...,...


In [106]:
def explode_cells(df):
    return df.loc[df.index.repeat(df['cells'])].reset_index(drop=True)


working_exploded=working.map_partitions(explode_cells)
working_exploded=working_exploded.reset_index(drop=True)
#^key! Otherwise we get garbage index.

In [107]:
working_exploded

,C(cell_type)[blood],C(cell_type)[brain],C(rep_id)[1],C(rep_id)[2],C(rep_id)[3],cre_id,cells,nb,zi,theta,r,sigmasquare,p
npartitions=2,,,,,,,,,,,,,
,int64,int64,int64,int64,int64,category[known],int64,float64,float64,float32,float32,float64,float64
,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...


In [108]:
import dask.array as da


In [109]:
r = working_exploded['r'].to_dask_array(lengths=True)
p = working_exploded['p'].to_dask_array(lengths=True)
nb_samples = da.random.negative_binomial(n=r, p=p, size=r.shape[0], chunks=r.chunks)


probabilities = working_exploded['zi'].to_dask_array(lengths=True)
bernoulli_trials = da.random.binomial(n=1, p=probabilities, size=probabilities.shape[0], chunks=probabilities.chunks)

zinb_samples=nb_samples * bernoulli_trials

working_exploded['zinb_sample'] = dd.from_dask_array(zinb_samples, columns='zinb_sample')

In [112]:
type(working_exploded.compute())

pandas.core.frame.DataFrame

In [8]:
cluster.close()